# Rice Supply LSTM Forecasting — Standalone Colab Notebook

Self-contained analysis notebook: dependencies are pinned and installed in-notebook, and the dataset is read from Google Drive via a configurable path (with a fallback search so you don't need a specific folder structure).

**Keeping this in sync:** this notebook duplicates the analysis from `RiceSupplyAnalysis_LSTM_Eval.ipynb`. If that notebook changes, this one needs to be regenerated/updated separately.

Run the cells below in order.

In [ ]:
# 1. Create a working directory
import os
os.makedirs("/content/rice_supply_analysis", exist_ok=True)
%cd /content/rice_supply_analysis

In [ ]:
%%writefile requirements.txt
pandas>=2.0.0
numpy>=1.26.0,<2.0.0
matplotlib>=3.7.0
statsmodels>=0.14.0
scikit-learn>=1.3.0
tensorflow>=2.15.0

In [ ]:
# 2. Install dependencies (Colab usually has these already; this pins versions for reproducibility elsewhere)
!pip install -q -r requirements.txt

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# 4. Resolve the CSV path on Drive
# Adjust this if your CSV lives somewhere else in your Google Drive
RICE_CSV_PATH = '/content/drive/MyDrive/ricesupply_2011-2025.csv'

import glob
from pathlib import Path

path = RICE_CSV_PATH
if not Path(path).exists():
    matches = glob.glob('/content/drive/MyDrive/**/ricesupply_2011-2025.csv', recursive=True)
    if matches:
        path = matches[0]
        print(f"RICE_CSV_PATH not found, using discovered file instead: {path}")
    else:
        raise FileNotFoundError(
            "Could not find ricesupply_2011-2025.csv anywhere in your Google Drive. "
            "Upload it to your Drive and/or update RICE_CSV_PATH above."
        )

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.metrics import mean_squared_error,mean_absolute_error
from math import sqrt
# holt winters
# single exponential smoothing
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
# double and triple exponential smoothing
from statsmodels.tsa.holtwinters import ExponentialSmoothing

DATA DETAILS

In [ ]:
df= pd.read_csv(path, index_col='Bulan', parse_dates=True)
df.index.freq='MS'
df.head()

In [ ]:
df.drop(columns='Unnamed: 0',inplace=True)

In [ ]:
df.head()

In [ ]:
df['Jateng'].plot(figsize=(12,6))

In [ ]:
df2=df.filter(items = ['Bulan','Cirebon'], axis = 1)
df2.head()

In [ ]:
results = seasonal_decompose(df2['Cirebon'])
results.plot();

MODEL 1 LAPIS LSTM

In [ ]:
train = df2.iloc[:135]
test = df2.iloc[135:]
len(df)

Kode itu artinya Anda membagi data df2 berdasarkan urutan baris:

train = df2.iloc[:140] → ambil baris indeks posisi 0 s.d. 139 (total 140 baris) untuk data latih.

test = df2.iloc[140:] → ambil baris indeks posisi 140 s.d. terakhir untuk data uji.

Catatan penting:

Ini bukan split acak (tidak shuffle). Cocok untuk time series / data berurutan.

Pastikan df2 sudah diurutkan (mis. berdasarkan tanggal) sebelum split, kalau konteksnya time series.

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train)
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)
scaled_train[:10]

In [ ]:
# define generator
n_input = 3 # Jumlah data sebelumnya yang digunakan untuk memprediksi 1 titik berikutnya
n_features = 1  # Jumlah fitur per timestep (dalam kasus ini: hanya 1, yaitu jumlah pasokan beras)

generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)
X,y = generator[0]
print(f'Given the Array: \n{X.flatten()}')
print(f'Predict this y: \n {y}')

chatgpt : Dengan scaled_train adalah data deret waktu yang sudah dinormalisasi (misalnya data pasokan beras per bulan), TimeseriesGenerator akan otomatis membuat:

X berisi 3 bulan terakhir

y adalah bulan berikutnya

In [ ]:
# We do the same thing, but now instead for 12 months
n_input = 12
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)

Jika data fluktuatif jangka pendek → Gunakan n_input = 3
Jika data musiman atau tren panjang → Gunakan n_input = 12


In [ ]:
# define model
model = Sequential() # Membuat model sekuensial—artinya layer akan ditambahkan satu per satu secara berurutan.
model.add(LSTM(100, activation='relu', input_shape=(n_input, n_features)))
model.add(Dense(1)) # Output layer: 1 neuron, karena kita hanya ingin memprediksi 1 nilai output (misal: jumlah pasokan bulan depan).
model.compile(optimizer='adam', loss='mse')

model.add(LSTM(100, activation='relu', input_shape=(n_input, n_features)))

Menambahkan LSTM layer dengan 100 unit (neuron).

activation='relu' → ReLU dipakai agar model bisa belajar non-linearitas. Catatan: default LSTM biasanya memakai tanh; pakai ReLU hanya jika sudah dicoba dan terbukti lebih baik.

input_shape=(n_input, n_features) → input ke LSTM berupa urutan sepanjang n_input (misalnya 12 bulan terakhir), masing-masing dengan n_features variabel (biasanya 1 untuk univariat).

model.compile(optimizer='adam', loss='mse')
optimizer='adam' → optimisasi adaptif yang sering dipakai karena cepat konvergen.

loss='mse' → Mean Squared Error, cocok untuk regresi seperti forecasting.

In [ ]:
model.summary()

In [ ]:
# fit model
model.fit(generator,epochs=50)

In [ ]:
loss_per_epoch = model.history.history['loss']
plt.plot(range(len(loss_per_epoch)),loss_per_epoch)

In [ ]:
last_train_batch = scaled_train[-12:]
last_train_batch = last_train_batch.reshape((1, n_input, n_features))
print(model.predict(last_train_batch))

In [ ]:
scaled_test[0]

In [ ]:
test_predictions = []

first_eval_batch = scaled_train[-n_input:]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(len(test)):

    # get the prediction value for the first batch
    current_pred = model.predict(current_batch)[0]

    # append the prediction into the array
    test_predictions.append(current_pred)

    # use the prediction to update the batch and remove the first value
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

In [ ]:
test_predictions

In [ ]:
test.head()

In [ ]:
true_predictions = scaler.inverse_transform(test_predictions)
test['Predictions'] = true_predictions

In [ ]:
test.plot(figsize=(14,5))

In [ ]:
# ======================================================
# EVALUASI DATA TRAINING
# ======================================================
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
from math import sqrt

train_pred = model.predict(generator, verbose=0)
train_actual = scaled_train[n_input:]

train_pred_inv = scaler.inverse_transform(train_pred)
train_actual_inv = scaler.inverse_transform(train_actual)

train_result = pd.DataFrame({
    "Actual": train_actual_inv.flatten(),
    "Prediction": train_pred_inv.flatten()
}, index=train.index[n_input:])

rmse = sqrt(mean_squared_error(train_result["Actual"], train_result["Prediction"]))
mae = mean_absolute_error(train_result["Actual"], train_result["Prediction"])
mape = np.mean(np.abs((train_result["Actual"]-train_result["Prediction"])/train_result["Actual"]))*100

print(train_result.head())
print(f"Training RMSE : {rmse:.2f}")
print(f"Training MAE  : {mae:.2f}")
print(f"Training MAPE : {mape:.2f}%")

plt.figure(figsize=(15,5))
plt.plot(train_result.index, train_result["Actual"], label="Actual")
plt.plot(train_result.index, train_result["Prediction"], label="Prediction")
plt.title("Training Data: Actual vs Prediction")
plt.legend(); plt.grid(True); plt.show()



In [ ]:
# ======================================================
# EVALUASI DATA TESTING
# ======================================================
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt

test_result = test.copy()

rmse = sqrt(mean_squared_error(test_result.iloc[:,0], test_result["Predictions"]))
mae = mean_absolute_error(test_result.iloc[:,0], test_result["Predictions"])
mape = np.mean(np.abs((test_result.iloc[:,0]-test_result["Predictions"])/test_result.iloc[:,0]))*100

print(test_result.head())
print(f"Testing RMSE : {rmse:.2f}")
print(f"Testing MAE  : {mae:.2f}")
print(f"Testing MAPE : {mape:.2f}%")

plt.figure(figsize=(15,5))
plt.plot(test_result.index, test_result.iloc[:,0], label="Actual")
plt.plot(test_result.index, test_result["Predictions"], label="Prediction")
plt.title("Testing Data: Actual vs Prediction")
plt.legend(); plt.grid(True); plt.show()



In [ ]:
rmse_model1=sqrt(mean_squared_error(test['Cirebon'],test['Predictions']))
print(f'Mean squared error : {rmse_model1}')

In [ ]:
test.head()

MODEL 2 LAPIS LSTM

In [ ]:
train = df2.iloc[:140]
test = df2.iloc[140:]
len(df)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train)
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)
scaled_train[:10]

In [ ]:
# define generator
n_input = 3
n_features = 1
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)
X,y = generator[0]
print(f'Given the Array: \n{X.flatten()}')
print(f'Predict this y: \n {y}')

In [ ]:
# We do the same thing, but now instead for 12 months
n_input = 12
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)

In [ ]:
# Membuat model dengan dua lapis LSTM
model2 = Sequential()
model2.add(LSTM(100, activation='relu', return_sequences=True, input_shape=(n_input, n_features)))
model2.add(LSTM(100, activation='relu'))
model2.add(Dense(1))
model2.compile(optimizer='adam', loss='mse')

In [ ]:
model2.summary()

In [ ]:
# fit model
model2.fit(generator,epochs=50)

In [ ]:
loss_per_epoch = model2.history.history['loss']
plt.plot(range(len(loss_per_epoch)),loss_per_epoch)

In [ ]:
last_train_batch = scaled_train[-12:]
last_train_batch = last_train_batch.reshape((1, n_input, n_features))
print(model2.predict(last_train_batch))

In [ ]:
last_train_batch.shape

In [ ]:
scaled_test[0]

In [ ]:
test_predictions = []

first_eval_batch = scaled_train[-n_input:]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(len(test)):

    # get the prediction value for the first batch
    current_pred = model2.predict(current_batch)[0]

    # append the prediction into the array
    test_predictions.append(current_pred)

    # use the prediction to update the batch and remove the first value
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

In [ ]:
test_predictions

In [ ]:
test.head()

In [ ]:
true_predictions = scaler.inverse_transform(test_predictions)
test['Predictions'] = true_predictions

In [ ]:
test.plot(figsize=(14,5))

In [ ]:
rmse_model2=sqrt(mean_squared_error(test['Cirebon'],test['Predictions']))
print(f'Mean squared error : {rmse_model2}')

In [ ]:
test.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import numpy as np

# Masukkan data (pastikan tab dipakai sebagai pemisah jika dari file Excel disalin ke .txt atau CSV)
data = {
    'Bulan': pd.date_range(start='1/1/2011', periods=172, freq='MS'),  # 172 bulan dari Jan 2011
    'Cirebon': [14991,18099,26227,26939,27555,21858,22739,20520,26754,24516,27071,21156,
                14485,12567,17589,23187,25573,18805,19139,15642,22940,21524,22177,15986,
                14606,10383,17416,23242,22237,19262,18092,17449,23699,25200,22962,17723,
                16536,10978,12354,19812,20924,17611,13630,25190,24325,24643,19430,17335,
                12163,6170,15342,24457,25672,27051,16846,25588,21895,24915,25046,23400,
                20473,16189,16469,20661,20788,19579,12505,23674,21476,22797,22302,21943,
                25310,19750,22475,16581,19186,11190,17765,22642,15060,15046,16116,10244,
                5620,9065,20735,18102,22722,14992,34803,22211,19987,22558,23381,17980,
                15854,8994,14759,22067,25446,14084,25055,20715,19181,21204,18256,21961,
                20625,13364,15759,23352,16826,21218,18476,26879,27953,24233,23559,26350,
                22325,16934,22229,21082,18923,22557,22979,22253,22298,18361,21973,15877,
                13791,12128,20437,26031,20150,27407,39520,20370,29929,25262,17969,14246,
                5816,3947,14223,13847,24487,13262,17382,14253,13662,20621,13659,9001,
                8779,5002,6183,14840,18351,17250,14723,13970,15768,18754,20977,20958,
                18314,16080,15429,15339]
}

df = pd.DataFrame(data)
df['Index'] = np.arange(len(df))

# Regresi linear
model = LinearRegression()
X = df[['Index']]
y = df['Cirebon']
model.fit(X, y)
df['Trend'] = model.predict(X)

# Visualisasi tren
plt.figure(figsize=(14, 6))
plt.plot(df['Bulan'], df['Cirebon'], label='Data Asli', marker='o', linewidth=1)
plt.plot(df['Bulan'], df['Trend'], label='Garis Tren Linear', color='red', linestyle='--')
plt.title('Analisis Tren Jumlah Cirebon (2011–2025)')
plt.xlabel('Tahun')
plt.ylabel('Jumlah')
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Tampilkan slope tren (koefisien)
print("Slope tren:", model.coef_[0])

## Notes

- **Data:** read from Google Drive via `RICE_CSV_PATH`, with a fallback search across your Drive if the file isn't at that exact location — no personal/course-specific folder path required.
- **Restarting:** if you restart the Colab runtime, re-run all cells from the top — the working directory is wiped (Drive itself is unaffected).
- **Updating this notebook:** since the source is duplicated rather than synced, any future fix to the original analysis notebook needs to be re-applied here too.